# 🇮🇹 Italian Influencer Community — Sarcasm-Aware NLP Pipeline

**What this notebook does:**
1. **Preprocessing** — emoji → Italian text, hashtag/mention cleaning, repeated char normalization
2. **Embeddings** — `paraphrase-multilingual-mpnet-base-v2` (768-dim, multilingual, social-media aware)
3. **Sentiment** — Two-layer: Italian BERT (fast, local) + Claude API (sarcasm correction, selective)
4. **Clustering** — UMAP dimensionality reduction + HDBSCAN topic discovery + Claude auto-labelling
5. **Visualizations** — Interactive charts: sentiment distribution, sarcasm breakdown, embedding scatter, cluster heatmap

**Runtime:** GPU recommended (T4 or better). Set `Runtime → Change runtime type → T4 GPU`.  
**Estimated time:** ~5 min on GPU, ~12 min on CPU.


## 📦 Cell 1 — Install dependencies

In [ ]:
%%capture install_output
!pip install \
    sentence-transformers>=2.7.0 \
    transformers>=4.40.0 \
    torch>=2.2.0 \
    anthropic>=0.26.0 \
    emoji>=2.11.0 \
    umap-learn>=0.5.6 \
    hdbscan>=0.8.38 \
    plotly>=5.20.0 \
    tqdm>=4.66.0

print("✅ All packages installed.")


In [ ]:
from google import genai
from google.genai import types


## 🔑 Cell 2 — Anthropic API Key

Paste your key below **or** use Google Colab Secrets (recommended for Enterprise):  
`Secrets panel (🔑 icon) → Add secret → Name: ANTHROPIC_API_KEY`


In [ ]:
# Install the necessary library if you haven't:
# pip install -U 'anthropic[vertex]'

from anthropic import AnthropicVertex

# Vertex AI does not use an 'api_key' parameter.
# It uses your GCP Project ID and a supported region (e.g., us-central1).
PROJECT_ID = "your-project-id"
REGION = "us-central1"

client = AnthropicVertex(project_id=PROJECT_ID, region=REGION)


## ⚙️ Cell 3 — Configuration
Tweak these settings for your specific influencer and community.

In [ ]:
# ── Influencer context ─────────────────────────────────────────────────────────
INFLUENCER_CONTEXT = (
    "Influencer italiana lifestyle/opinioni, community molto ironica e sarcastica, "
    "usa humor nero e riferimenti alla cultura pop italiana"
)

# ── Embedding model ────────────────────────────────────────────────────────────
EMBEDDING_MODEL    = "paraphrase-multilingual-mpnet-base-v2"
EMBEDDING_BATCH_SIZE = 64

# ── Sentiment model ────────────────────────────────────────────────────────────
SENTIMENT_MODEL    = "neuraly/bert-base-italian-cased-sentiment"
SENTIMENT_BATCH_SIZE = 32
SENTIMENT_LABELS   = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}

# ── Sarcasm detection ──────────────────────────────────────────────────────────
# Claude is called only when base confidence < threshold OR sarcasm keyword found
SARCASM_CONFIDENCE_THRESHOLD = 0.75
SARCASM_KEYWORDS_IT = [
    "certo", "ovviamente", "naturalmente", "che sorpresa", "geniale",
    "bravo", "bravissimo", "grande", "magari", "eh già", "wow",
    "incredibile", "fantastico", "perfetto", "che bella", "meno male",
    "non ci credo", "ci mancherebbe", "come no", "ma va",
]

# ── Claude settings ────────────────────────────────────────────────────────────
CLAUDE_MODEL      = "claude-sonnet-4-20250514"
CLAUDE_MAX_TOKENS = 256

# ── Clustering ─────────────────────────────────────────────────────────────────
UMAP_N_NEIGHBORS          = 15
UMAP_MIN_DIST             = 0.1
UMAP_N_COMPONENTS         = 5      # internal; 2D projection done separately for viz
HDBSCAN_MIN_CLUSTER_SIZE  = 5      # lower = finer clusters; increase for large datasets
HDBSCAN_MIN_SAMPLES       = 3

print("✅ Configuration ready.")


✅ Configuration ready.


## 📚 Cell 4 — Imports

In [ ]:
from __future__ import annotations
import os, re, json, unicodedata, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from tqdm.notebook import tqdm

import emoji
import anthropic
import umap
import hdbscan
from sentence_transformers import SentenceTransformer
from transformers import pipeline as hf_pipeline
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Imports OK  |  Device: {device.upper()}")


✅ Imports OK  |  Device: CPU


## 🧹 Cell 5 — Preprocessing utilities

In [ ]:
def demojize_italian(text: str) -> str:
    """Convert emoji to Italian text descriptions wrapped in brackets [emoji: ...]."""
    def _replace(match):
        em = match.group()
        alias = emoji.demojize(em, language="it")
        if alias == em:
            alias = emoji.demojize(em, language="en")
        alias = alias.strip(":").replace("_", " ")
        return f" {alias} "

    pattern = re.compile(
        "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF\U0001F900-\U0001F9FF"
        "\U00002600-\U000026FF]+",
        flags=re.UNICODE,
    )
    return pattern.sub(_replace, text)


def normalize_repeated_chars(text: str, max_repeat: int = 2) -> str:
    """'bravoooooo' → 'bravoo' — preserves Italian double letters."""
    return re.sub(r"(.)\1{" + str(max_repeat) + r",}", r"\1" * max_repeat, text)


def preprocess_for_bert_and_embeddings(text: str) -> str:
    """
    Preprocessing for BERT sentiment & embeddings.
    Strips emoji entirely — BERT needs clean Italian tokens.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFC", text)
    # Strip emoji entirely (don't convert to text)
    text = re.sub(
        r"[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
        r"\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
        r"\U00002700-\U000027BF\U0001F900-\U0001F9FF"
        r"\U00002600-\U000026FF]+",
        " ", text, flags=re.UNICODE
    )
    text = re.sub(r"https?://\S+|www\.\S+", " [URL] ", text)   # URLs
    text = re.sub(r"@\w+", " [UTENTE] ", text)                    # mentions
    text = re.sub(r"#(\w+)", r"\1", text)                        # hashtags
    text = normalize_repeated_chars(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_for_claude(text: str) -> str:
    """
    Preprocessing for Claude sarcasm detection.
    Converts emoji to [emoji:name] format — preserves irony signals.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFC", text)
    text = demojize_italian(text)  # Convert emoji to [emoji:name] format
    text = re.sub(r"https?://\S+|www\.\S+", " [URL] ", text)   # URLs
    text = re.sub(r"@\w+", " [UTENTE] ", text)                    # mentions
    text = re.sub(r"#(\w+)", r"\1", text)                        # hashtags
    text = normalize_repeated_chars(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_batch_bert_embeddings(texts: list[str]) -> list[str]:
    """Batch preprocess for BERT and embeddings (strip emoji)."""
    return [preprocess_for_bert_and_embeddings(t) for t in texts]


def preprocess_batch_claude(texts: list[str]) -> list[str]:
    """Batch preprocess for Claude (emoji as [emoji:name])."""
    return [preprocess_for_claude(t) for t in texts]


# Quick smoke test
_test = "Ma certo, geniale come al solito 😤 #italia @utente https://t.co/xxx"
print(f"Input:                {_test}")
print(f"BERT/Embeddings:      {preprocess_for_bert_and_embeddings(_test)}")
print(f"Claude:               {preprocess_for_claude(_test)}")
print("✅ Dual preprocessing OK")


Input:  Ma certo, geniale come al solito 😤 #italia @utente https://t.co/xxx
Output: Ma certo, geniale come al solito faccina che sbuffa italia [UTENTE] [URL]
✅ Preprocessing OK


## 🧠 Cell 6 — Embedding model
Loads `paraphrase-multilingual-mpnet-base-v2` (~420 MB). This may take 1–2 minutes on first run.

In [ ]:
class ItalianEmbedder:
    """768-dim multilingual sentence embeddings optimised for Italian social media."""

    def __init__(self, model_name=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH_SIZE):
        self.batch_size = batch_size
        print(f"[Embedder] Loading {model_name} …")
        self.model = SentenceTransformer(model_name, device=device)
        print(f"[Embedder] Ready on {device.upper()}  |  dim=768")

    def encode(self, texts: list[str], normalize: bool = True, show_progress: bool = True) -> np.ndarray:
        if not texts:
            return np.empty((0, 768), dtype=np.float32)
        vecs = self.model.encode(
            texts,
            batch_size=self.batch_size,
            normalize_embeddings=normalize,
            show_progress_bar=show_progress,
            convert_to_numpy=True,
        )
        return vecs.astype(np.float32)

    def top_similar(self, query, corpus_vectors, corpus_texts, top_k=5):
        if isinstance(query, str):
            query = self.encode([query], show_progress=False)[0]
        scores = corpus_vectors @ query
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(float(scores[i]), corpus_texts[i]) for i in top_idx]


embedder = ItalianEmbedder()


[Embedder] Loading paraphrase-multilingual-mpnet-base-v2 …


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Embedder] Ready on CPU  |  dim=768


## 💬 Cell 7 — Sentiment analyser (two-layer)
Layer A: Italian BERT (local, fast). Layer B: Claude API (sarcasm correction, called selectively).

In [ ]:
SARCASM_PROMPT = """Sei un esperto di linguistica e analisi del sentiment sui social media italiani.
Analizza questo testo di un post/commento sui social media:

TESTO: "{text}"

Il sentiment iniziale rilevato automaticamente è: {base_sentiment} (confidenza: {confidence:.0%})

Il tuo compito:
1. Determina se c'è sarcasmo, ironia o umorismo che inverte il sentiment reale.
2. Fornisci il sentiment REALE del testo.

Rispondi SOLO con questo JSON (nessun testo aggiuntivo):
{{
  "is_sarcastic": true/false,
  "real_sentiment": "positive" | "neutral" | "negative",
  "confidence": 0.0-1.0,
  "reason": "breve spiegazione in italiano (max 15 parole)"
}}"""


@dataclass
class SentimentResult:
    text: str
    processed_text_bert: str = ""
    processed_text_claude: str = ""
    base_sentiment: str = ""
    base_confidence: float = 0.0
    is_sarcastic: bool = False
    final_sentiment: str = ""
    final_confidence: float = 0.0
    sarcasm_reason: str = ""
    claude_called: bool = False

    def to_dict(self):
        return {
            "text": self.text,
            "base_sentiment": self.base_sentiment,
            "base_confidence": round(self.base_confidence, 4),
            "is_sarcastic": self.is_sarcastic,
            "final_sentiment": self.final_sentiment,
            "final_confidence": round(self.final_confidence, 4),
            "sarcasm_reason": self.sarcasm_reason,
            "claude_called": self.claude_called,
        }


class ItalianSentimentAnalyser:
    """Two-layer sarcasm-aware Italian sentiment analyser using Anthropic on Vertex AI.
    
    Dual preprocessing:
    - BERT/embeddings: preprocess_for_bert_and_embeddings() [strips emoji]
    - Claude sarcasm check: preprocess_for_claude() [keeps emoji as [emoji:name]]
    """

    def __init__(self):
        print(f"[Sentiment] Loading {SENTIMENT_MODEL} …")
        self._clf = hf_pipeline(
            "text-classification",
            model=SENTIMENT_MODEL,
            truncation=True,
            max_length=512,
            device=0 if device == "cuda" else -1,
        )
        # Use the Vertex AI client initialized in Cell 2
        self._claude = client
        self._kw_re = re.compile(
            r"\b(" + "|".join(re.escape(k) for k in SARCASM_KEYWORDS_IT) + r")\b",
            re.IGNORECASE,
        )
        print("[Sentiment] Ready (Vertex AI Layer Active).")

    def _base(self, texts_bert):
        """Run BERT sentiment on preprocessed texts (emoji stripped)."""
        out = []
        for i in range(0, len(texts_bert), SENTIMENT_BATCH_SIZE):
            for p in self._clf(texts_bert[i:i+SENTIMENT_BATCH_SIZE]):
                out.append((SENTIMENT_LABELS.get(p["label"], p["label"]), float(p["score"])))
        return out

    def _needs_check(self, text_original, conf):
        """Check if Claude should verify this text for sarcasm."""
        return conf < SARCASM_CONFIDENCE_THRESHOLD or bool(self._kw_re.search(text_original))

    def _detect_sarcasm(self, text_claude, base_sent, conf):
        """Call Claude to detect sarcasm using Claude-preprocessed text (with [emoji:name])."""
        prompt = SARCASM_PROMPT.format(text=text_claude, base_sentiment=base_sent, confidence=conf)
        try:
            # Vertex AI call syntax matches the Anthropic SDK
            resp = self._claude.messages.create(
                model=CLAUDE_MODEL, max_tokens=CLAUDE_MAX_TOKENS,
                system=(
                    "Sei un esperto di NLP specializzato nell'analisi del sentiment "
                    "e del sarcasmo in italiano sui social media. "
                    "Rispondi SEMPRE e SOLO in formato JSON valido."
                ),
                messages=[{"role": "user", "content": prompt}],
            )
            raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", resp.content[0].text.strip(), flags=re.MULTILINE)
            d = json.loads(raw)
            return bool(d.get("is_sarcastic", False)), str(d.get("real_sentiment", base_sent)), float(d.get("confidence", conf)), str(d.get("reason", ""))
        except Exception as e:
            return False, base_sent, conf, f"[err: {e}]"

    def analyse(self, original_texts, show_progress=True):
        """Analyse sentiment with sarcasm detection using dual preprocessing.
        
        Args:
            original_texts: List of original (unprocessed) texts
            show_progress: Show progress bar
        
        Returns:
            List of SentimentResult objects
        """
        # Preprocess: two paths
        texts_bert = preprocess_batch_bert_embeddings(original_texts)      # For BERT
        texts_claude = preprocess_batch_claude(original_texts)              # For Claude (if needed)
        
        print("[Sentiment] Running Italian BERT (Layer A) …")
        base = self._base(texts_bert)
        
        results, n_claude = [], 0
        it = zip(original_texts, texts_bert, texts_claude, base)
        if show_progress:
            it = tqdm(it, total=len(original_texts), desc="Sarcasm check (Layer B, selective)")
        
        for orig, proc_bert, proc_claude, (bl, bc) in it:
            r = SentimentResult(
                text=orig, 
                processed_text_bert=proc_bert,
                processed_text_claude=proc_claude,
                base_sentiment=bl, 
                base_confidence=bc
            )
            
            # Selective Claude call: only if confidence low OR sarcasm keyword detected
            if self._needs_check(orig, bc):
                r.is_sarcastic, r.final_sentiment, r.final_confidence, r.sarcasm_reason = \
                    self._detect_sarcasm(proc_claude, bl, bc)
                r.claude_called = True
                n_claude += 1
            else:
                r.final_sentiment, r.final_confidence = bl, bc
            
            results.append(r)
        
        pct = 100 * n_claude / max(len(results), 1)
        print(f"[Sentiment] Done. Claude called for {n_claude}/{len(results)} posts ({pct:.1f}%)")
        return results

sentiment_analyser = ItalianSentimentAnalyser()


[Sentiment] Loading neuraly/bert-base-italian-cased-sentiment …


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[Sentiment] Ready (Vertex AI Layer Active).


## 🗂️ Cell 8 — Topic clustering (UMAP + HDBSCAN)

In [ ]:
class TopicClusterer:
    """UMAP dimensionality reduction + HDBSCAN clustering + Claude auto-labelling."""

    def __init__(self):
        self._umap_full = umap.UMAP(
            n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
            n_components=UMAP_N_COMPONENTS, metric="cosine", random_state=42,
        )
        self._umap_2d = umap.UMAP(
            n_neighbors=UMAP_N_NEIGHBORS, min_dist=0.05,
            n_components=2, metric="cosine", random_state=42,
        )
        self._hdbscan = hdbscan.HDBSCAN(
            min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=HDBSCAN_MIN_SAMPLES,
            metric="euclidean", cluster_selection_method="eom",
        )
        self._claude = anthropic.Anthropic()

    def fit(self, embeddings):
        print("[Cluster] UMAP 768→5D …")
        reduced_5d = self._umap_full.fit_transform(embeddings)
        print("[Cluster] UMAP 768→2D (for visualization) …")
        reduced_2d = self._umap_2d.fit_transform(embeddings)
        print("[Cluster] HDBSCAN …")
        labels = self._hdbscan.fit_predict(reduced_5d)
        n_c = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        print(f"[Cluster] {n_c} clusters found | {n_noise} noise points ({100*n_noise/len(labels):.1f}%)")
        return labels, reduced_2d

    def cluster_summary(self, labels, texts, top_k=5):
        summary = {}
        for cid in sorted(set(labels)):
            if cid == -1: continue
            mask = labels == cid
            summary[cid] = {
                "size": int(mask.sum()),
                "representative_posts": [t for t, m in zip(texts, mask) if m][:top_k],
                "label": None, "description": "",
            }
        return summary

    def auto_label_clusters(self, summary, context=""):
        print(f"[Cluster] Auto-labelling {len(summary)} clusters with Claude …")
        for cid, info in summary.items():
            posts = "\n".join(f"- {p}" for p in info["representative_posts"])
            prompt = (
                f"Sei un analista di social media italiano.\n"
                + (f"Contesto influencer: {context}\n" if context else "")
                + f"\nPost del cluster {cid}:\n{posts}\n\n"
                "Fornisci etichetta (max 5 parole) e descrizione (1 frase).\n"
                'Rispondi SOLO in JSON: {"label": "...", "description": "..."}'
            )
            try:
                resp = self._claude.messages.create(
                    model=CLAUDE_MODEL, max_tokens=CLAUDE_MAX_TOKENS,
                    messages=[{"role": "user", "content": prompt}],
                )
                raw = re.sub(r"^```(?:json)?\s*|\s*```$", "", resp.content[0].text.strip(), flags=re.MULTILINE)
                d = json.loads(raw)
                info["label"] = d.get("label", f"Cluster {cid}")
                info["description"] = d.get("description", "")
            except Exception as e:
                info["label"] = f"Cluster {cid}"
                info["description"] = str(e)
        return summary


clusterer = TopicClusterer()
print("✅ Clusterer ready.")


✅ Clusterer ready.


## 📥 Cell 8 — Load Instagram comments from Google Cloud Storage


In [ ]:
from google.cloud import storage
import pandas as pd

# ── GCS Configuration ──────────────────────────────────────────────────────────
project_id = 'gen-lang-client-0792749758'
bucket_name = 'afb_showreel'
base_path = f"gs://{bucket_name}/modified_comments_parquet/"

# ── Connect to GCS ─────────────────────────────────────────────────────────────
client = storage.Client(project=project_id)
bucket = client.bucket(bucket_name)

print(f"Connecting to gs://{bucket_name}...")
blobs = bucket.list_blobs()
file_count = sum(1 for _ in blobs)
print(f"✅ Found {file_count} files in bucket.")

# ── Load Instagram Comments ───────────────────────────────────────────────────
print("\n[Data] Loading ig_comments_cleaned.parquet...")
ig_comments = pd.read_parquet(base_path + "ig_comments_cleaned.parquet")
print(f"Loaded: {len(ig_comments)} rows")
print(f"Columns: {list(ig_comments.columns)}")

# Data quality check
print(f"\n[Data Quality] Missing values before processing:")
print(ig_comments.isnull().sum())

# ── Sample for speed ───────────────────────────────────────────────────────────
print(f"\n[Data] Sampling 2000 comments for speed...")
ig_comments = ig_comments.sample(n=min(2000, len(ig_comments)), random_state=42)
print(f"After sampling: {len(ig_comments)} rows")

# ── Convert to RAW_POSTS (list of comment texts) ──────────────────────────────
RAW_POSTS = ig_comments['text'].fillna("").tolist()
print(f"\n✅ Ready to analyze {len(RAW_POSTS)} posts")


Contents of gs://afb_showreel:
 - 2519586145507999744/content.html (1.22 MB)
 - 2519586145507999744/content.ipynb (0.66 MB)
 - 3488985965299499008/content.html (1.22 MB)
 - 3488985965299499008/content.ipynb (0.66 MB)
 - Output/ (0.00 MB)
 - Output/Comments_with_LLM/fb_comments_with_llm.parquet (36.56 MB)
 - Output/Comments_with_LLM/ig_comments_with_llm.parquet (43.90 MB)
 - Output/Comments_with_LLM/tk_comments_with_llm.parquet (1.48 MB)
 - Output/Comments_with_UID/fb_comments_uid.parquet (33.49 MB)
 - Output/Comments_with_UID/ig_comments_uid.parquet (40.61 MB)
 - Output/Comments_with_UID/tk_comments_uid.parquet (1.44 MB)
 - Output/Matches/all_fuzzy_matches_sample.parquet (0.06 MB)
 - Output/Matches/best_fuzzy_matches_sample.parquet (0.04 MB)
 - Output/Matches/perfect_100_matches_sample.parquet (0.01 MB)
 - Output/Universal_id/ (0.00 MB)
 - Output/Universal_id/6185533834373627904/content.ipynb (0.07 MB)
 - Output/Universal_id/identity_registry.parquet (0.02 MB)
 - YTcomments_1_cleaned.p

## 🚀 Cell 9 — Run the full pipeline


In [ ]:
print("=" * 60)
print(f"  Italian Social Media Analyser — {len(RAW_POSTS)} posts")
print("=" * 60)

# 1. Preprocessing for embeddings and BERT
print("\n[1/4] Preprocessing for embeddings …")
processed_texts_bert = preprocess_batch_bert_embeddings(RAW_POSTS)
print(f"      {len(processed_texts_bert)} texts preprocessed (emoji stripped).")

# 2. Embeddings (using BERT-preprocessed texts)
print("\n[2/4] Generating embeddings …")
embeddings = embedder.encode(processed_texts_bert, show_progress=True)
print(f"      Embedding matrix: {embeddings.shape}")

# 3. Sentiment + sarcasm (dual preprocessing internally in the analyzer)
print("\n[3/4] Sentiment analysis …")
sentiment_results = sentiment_analyser.analyse(RAW_POSTS, show_progress=True)

# 4. Clustering
print("\n[4/4] Topic clustering …")
RUN_CLUSTERING = len(RAW_POSTS) >= 10
if RUN_CLUSTERING:
    cluster_labels, reduced_2d = clusterer.fit(embeddings)
    cluster_summary = clusterer.cluster_summary(cluster_labels, RAW_POSTS)
    cluster_summary = clusterer.auto_label_clusters(cluster_summary, INFLUENCER_CONTEXT)
else:
    cluster_labels  = np.full(len(RAW_POSTS), -1, dtype=int)
    reduced_2d      = np.zeros((len(RAW_POSTS), 2))
    cluster_summary = {}
    print("      Skipped (< 10 posts).")

print("\n✅ Pipeline complete.")


## 🗃️ Cell 11 — Build results DataFrame

In [ ]:
rows = []
for i, (text, sent, clabel) in enumerate(zip(RAW_POSTS, sentiment_results, cluster_labels)):
    rows.append({
        "idx": i,
        "text": text,
        "processed_text_bert": sent.processed_text_bert,
        "processed_text_claude": sent.processed_text_claude,
        "base_sentiment": sent.base_sentiment,
        "base_confidence": round(sent.base_confidence, 4),
        "is_sarcastic": sent.is_sarcastic,
        "final_sentiment": sent.final_sentiment,
        "final_confidence": round(sent.final_confidence, 4),
        "sarcasm_reason": sent.sarcasm_reason,
        "claude_called": sent.claude_called,
        "cluster_id": int(clabel),
        "cluster_label": (
            cluster_summary.get(int(clabel), {}).get("label", "noise")
            if clabel != -1 else "noise"
        ),
        "umap_x": float(reduced_2d[i, 0]) if RUN_CLUSTERING else 0.0,
        "umap_y": float(reduced_2d[i, 1]) if RUN_CLUSTERING else 0.0,
    })

df = pd.DataFrame(rows)

# ── Summary stats ──────────────────────────────────────────────────────────────
n = len(df)
n_sarc = df["is_sarcastic"].sum()
n_claude = df["claude_called"].sum()
n_clusters = df[df["cluster_id"] != -1]["cluster_id"].nunique()

print(f"{'='*55}")
print(f"  RESULTS SUMMARY — {n} posts")
print(f"{'='*55}")
print(f"\nSentiment distribution (sarcasm-corrected):")
for lbl, cnt in df["final_sentiment"].value_counts().items():
    bar = "█" * int(30 * cnt / n)
    print(f"  {lbl:<10} {cnt:>4}  {bar}  ({100*cnt/n:.1f}%)")

print(f"\nSarcasm:")
print(f"  Sarcastic posts   {n_sarc:>4}  ({100*n_sarc/n:.1f}%)")
print(f"  Claude API calls  {n_claude:>4}  ({100*n_claude/n:.1f}%)")
print(f"\nTopics:   {n_clusters} clusters found")
for cid, info in sorted(cluster_summary.items()):
    print(f"  [{cid}] {info.get('label','?')} ({info['size']} posts)")
print(f"{'='*55}")

# Data quality checks per instructions
print(f"\n[Data Quality] Checking for missing values:")
print(df.isnull().sum())

df.head(10)


## 📊 Cell 12 — Visualization: Sentiment distribution

In [ ]:
sent_counts = df["final_sentiment"].value_counts().reset_index()
sent_counts.columns = ["sentiment", "count"]

color_map = {"positive": "#1D9E75", "neutral": "#888780", "negative": "#D85A30"}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Final sentiment (sarcasm-corrected)", "Base vs corrected"),
    specs=[[{"type": "bar"}, {"type": "bar"}]]
)

# Left: final sentiment
fig.add_trace(
    go.Bar(
        x=sent_counts["sentiment"],
        y=sent_counts["count"],
        marker_color=[color_map.get(s, "#888") for s in sent_counts["sentiment"]],
        text=sent_counts["count"],
        textposition="outside",
        name="Final",
    ),
    row=1, col=1,
)

# Right: base vs corrected comparison
base_counts = df["base_sentiment"].value_counts()
final_counts = df["final_sentiment"].value_counts()
labels = ["negative", "neutral", "positive"]
fig.add_trace(
    go.Bar(name="Base (BERT)", x=labels,
           y=[base_counts.get(l, 0) for l in labels],
           marker_color=["#F09595", "#D3D1C7", "#9FE1CB"], opacity=0.7),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(name="Corrected (Claude)", x=labels,
           y=[final_counts.get(l, 0) for l in labels],
           marker_color=[color_map[l] for l in labels]),
    row=1, col=2,
)

fig.update_layout(
    title="Italian Influencer Community — Sentiment Analysis",
    barmode="group", height=420,
    font=dict(family="Arial", size=13),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_yaxes(gridcolor="#eee")
fig.show()


## 🎭 Cell 13 — Visualization: Sarcasm breakdown

In [ ]:
# Posts where sarcasm was detected and sentiment was CORRECTED
corrected = df[df["is_sarcastic"] & (df["base_sentiment"] != df["final_sentiment"])].copy()
not_corrected = df[df["is_sarcastic"] & (df["base_sentiment"] == df["final_sentiment"])].copy()
no_sarcasm = df[~df["is_sarcastic"]].copy()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Sarcasm detection ({n_sarc}/{n} posts flagged)",
        "Sentiment flip — base → corrected",
    ),
    specs=[[{"type": "pie"}, {"type": "sankey"}]],
)

# Pie
fig.add_trace(
    go.Pie(
        labels=["No sarcasm", "Sarcastic (corrected)", "Sarcastic (no change)"],
        values=[len(no_sarcasm), len(corrected), len(not_corrected)],
        marker_colors=["#1D9E75", "#D85A30", "#EF9F27"],
        hole=0.45,
        textinfo="label+percent",
    ),
    row=1, col=1,
)

# Sankey: base → final transitions
if len(corrected) > 0:
    labels_s = ["base neg", "base neu", "base pos", "final neg", "final neu", "final pos"]
    label_idx = {l: i for i, l in enumerate(labels_s)}
    sources, targets, values_s = [], [], []
    for (bs, fs), grp in df.groupby(["base_sentiment", "final_sentiment"]):
        src = f"base {bs[:3]}"
        tgt = f"final {fs[:3]}"
        if src in label_idx and tgt in label_idx:
            sources.append(label_idx[src])
            targets.append(label_idx[tgt])
            values_s.append(len(grp))
    fig.add_trace(
        go.Sankey(
            node=dict(
                label=labels_s,
                color=["#F09595","#D3D1C7","#9FE1CB","#A32D2D","#5F5E5A","#1D9E75"],
                pad=15, thickness=20,
            ),
            link=dict(source=sources, target=targets, value=values_s),
        ),
        row=1, col=2,
    )

fig.update_layout(height=420, title="Sarcasm Detection & Sentiment Correction",
                  font=dict(family="Arial", size=12))
fig.show()

if len(corrected) > 0:
    print("\n🎭 Posts where sarcasm flipped the sentiment:")
    for _, row in corrected.head(5).iterrows():
        print(f"  [{row['base_sentiment']} → {row['final_sentiment']}] {row['text'][:80]}")
        print(f"  Reason: {row['sarcasm_reason']}\n")


## 🗺️ Cell 14 — Visualization: Embedding space (UMAP 2D)

In [ ]:
if RUN_CLUSTERING:
    df_plot = df.copy()
    df_plot["label_short"] = df_plot["text"].str[:50] + "…"
    df_plot["cluster_str"] = df_plot["cluster_label"].astype(str)

    fig = px.scatter(
        df_plot, x="umap_x", y="umap_y",
        color="cluster_str",
        symbol="final_sentiment",
        symbol_map={"positive": "circle", "neutral": "square", "negative": "x"},
        hover_data={"label_short": True, "final_sentiment": True,
                    "is_sarcastic": True, "umap_x": False, "umap_y": False},
        title="Post Embedding Space — UMAP 2D Projection",
        labels={"cluster_str": "Topic cluster", "umap_x": "UMAP-1", "umap_y": "UMAP-2"},
        height=560,
    )

    # Mark sarcastic posts with a ring
    sarc_df = df_plot[df_plot["is_sarcastic"]]
    if len(sarc_df) > 0:
        fig.add_trace(go.Scatter(
            x=sarc_df["umap_x"], y=sarc_df["umap_y"],
            mode="markers",
            marker=dict(symbol="circle-open", size=16, color="black", line=dict(width=2)),
            name="⚠ Sarcastic",
            hoverinfo="skip",
        ))

    fig.update_layout(
        plot_bgcolor="white", paper_bgcolor="white",
        font=dict(family="Arial", size=12),
        legend=dict(itemsizing="constant"),
    )
    fig.update_xaxes(gridcolor="#eee", zeroline=False)
    fig.update_yaxes(gridcolor="#eee", zeroline=False)
    fig.show()
    print("Circles with ring outline = sarcastic posts")
else:
    print("Clustering was skipped — no UMAP visualization available.")


## 🔥 Cell 15 — Visualization: Cluster × Sentiment heatmap

In [ ]:
if RUN_CLUSTERING and len(cluster_summary) > 0:
    pivot = pd.crosstab(df["cluster_label"], df["final_sentiment"])
    # Ensure all sentiment columns present
    for col in ["negative", "neutral", "positive"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot = pivot[["negative", "neutral", "positive"]]

    fig = go.Figure(data=go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale=[[0,"#E1F5EE"],[0.5,"#EF9F27"],[1,"#D85A30"]],
        text=pivot.values,
        texttemplate="%{text}",
        textfont={"size": 14},
        hovertemplate="Cluster: %{y}<br>Sentiment: %{x}<br>Posts: %{z}<extra></extra>",
    ))
    fig.update_layout(
        title="Cluster × Sentiment Heatmap",
        xaxis_title="Final sentiment",
        yaxis_title="Topic cluster",
        height=max(350, 80 * len(pivot) + 100),
        font=dict(family="Arial", size=12),
        plot_bgcolor="white",
    )
    fig.show()

    # Sarcasm rate per cluster
    sarc_rate = df.groupby("cluster_label")["is_sarcastic"].mean().sort_values(ascending=False)
    fig2 = go.Figure(go.Bar(
        x=sarc_rate.index, y=(sarc_rate * 100).round(1),
        marker_color="#EF9F27",
        text=(sarc_rate * 100).round(1).astype(str) + "%",
        textposition="outside",
    ))
    fig2.update_layout(
        title="Sarcasm rate per topic cluster",
        yaxis_title="% sarcastic posts",
        xaxis_title="Cluster",
        height=380,
        plot_bgcolor="white",
        yaxis=dict(gridcolor="#eee"),
    )
    fig2.show()
else:
    print("Clustering was skipped — no heatmap available.")


## 📋 Cell 16 — Detailed results table

In [ ]:
from IPython.display import display, HTML

def highlight_row(row):
    styles = [""] * len(row)
    if row["is_sarcastic"]:
        styles = ["background-color: #FFF3CD"] * len(row)
    if row["final_sentiment"] == "negative":
        styles[df.columns.get_loc("final_sentiment")] = "color: #A32D2D; font-weight: bold"
    elif row["final_sentiment"] == "positive":
        styles[df.columns.get_loc("final_sentiment")] = "color: #0F6E56; font-weight: bold"
    return styles

display_cols = ["text", "final_sentiment", "base_sentiment", "is_sarcastic",
                "final_confidence", "sarcasm_reason", "cluster_label"]
styled = (
    df[display_cols]
    .style
    .apply(highlight_row, axis=1)
    .format({"final_confidence": "{:.2f}"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#f8f9fa"), ("font-weight", "600")]},
        {"selector": "td", "props": [("font-size", "12px"), ("max-width", "300px"),
                                      ("overflow", "hidden"), ("text-overflow", "ellipsis")]},
    ])
    .set_caption("🇮🇹 Italian Influencer Community — Full Results  |  Yellow = sarcasm detected")
)
display(styled)


## 🔍 Cell 17 — Semantic similarity search
Find posts semantically similar to a query in Italian.

In [ ]:
# ── Change this query to explore the embedding space ──────────────────────────
QUERY = "problemi e critiche"

hits = embedder.top_similar(QUERY, embeddings, RAW_POSTS, top_k=5)

print(f"Top 5 posts most similar to: \"{QUERY}\"\n")
for rank, (score, text) in enumerate(hits, 1):
    idx = RAW_POSTS.index(text)
    r = df.iloc[idx]
    sarc_tag = " ⚠ SARCASMO" if r["is_sarcastic"] else ""
    print(f"  {rank}. [{score:.3f}] {r['final_sentiment'].upper()}{sarc_tag}")
    print(f"     {text[:100]}")
    if r["sarcasm_reason"]:
        print(f"     → {r['sarcasm_reason']}")
    print()


## 💾 Cell 18 — Save results

In [ ]:
import datetime

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# ── Save CSV ───────────────────────────────────────────────────────────────────
csv_path = f"/content/italian_sentiment_results_{ts}.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✅ CSV saved: {csv_path}")

# ── Save embeddings ────────────────────────────────────────────────────────────
emb_path = f"/content/embeddings_{ts}.npy"
np.save(emb_path, embeddings)
print(f"✅ Embeddings saved: {emb_path}")

# ── Save cluster summary ───────────────────────────────────────────────────────
if cluster_summary:
    import json as _json
    cs_path = f"/content/cluster_summary_{ts}.json"
    with open(cs_path, "w", encoding="utf-8") as f:
        _json.dump({str(k): {kk: vv for kk, vv in v.items() if kk != "representative_posts"}
                    for k, v in cluster_summary.items()}, f, ensure_ascii=False, indent=2)
    print(f"✅ Cluster summary saved: {cs_path}")

# ── Download to local machine ──────────────────────────────────────────────────
try:
    from google.colab import files
    files.download(csv_path)
    print("📥 CSV download triggered.")
except Exception:
    print(f"(Not in interactive Colab session — find your files in /content/)")


## 🧪 Cell 19 — Quick single-post analysis
Test any Italian text interactively.

In [ ]:
# ── Change this to any Italian text ───────────────────────────────────────────
TEST_TEXT = "Ma certo, ovviamente ha funzionato tutto alla perfezione come sempre 🙄"

proc_bert = preprocess_for_bert_and_embeddings(TEST_TEXT)
proc_claude = preprocess_for_claude(TEST_TEXT)
emb  = embedder.encode([proc_bert], show_progress=False)
res  = sentiment_analyser.analyse([TEST_TEXT], show_progress=False)[0]

print("─" * 60)
print(f"Input:              {TEST_TEXT}")
print("─" * 60)
print(f"BERT preproc:       {proc_bert}")
print(f"Claude preproc:     {proc_claude}")
print("─" * 60)
print(f"Base sentiment:     {res.base_sentiment}  (conf: {res.base_confidence:.2f})")
print(f"Sarcastic:          {'YES ⚠' if res.is_sarcastic else 'no'}")
print(f"Final sentiment:    {res.final_sentiment}  (conf: {res.final_confidence:.2f})")
if res.sarcasm_reason:
    print(f"Reason:             {res.sarcasm_reason}")
if res.claude_called:
    print(f"Claude called:      Yes (Layer B)")
print("─" * 60)
